# Dense Ollama Retriever Step-by-Step

`dense_ollama_retriever.py` 안의 함수들을 하나씩 실행하면서 중간 결과를 확인하는 노트북입니다.

확인할 흐름:

1. 프로젝트 경로 설정
2. Ollama 연결 확인
3. chunk 파일 로드
4. query와 chunk를 embedding으로 변환
5. cosine similarity 직접 계산
6. embedding cache 확인
7. dense search 실행


## 1. 프로젝트 경로 설정

노트북에서 `scripts/` 폴더의 코드를 import하기 위해 프로젝트 루트를 Python path에 추가합니다.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('c:/학부연구생/wikiRAG')

## 2. dense_ollama_retriever 함수 import

이번 노트북에서는 파일 안의 함수를 직접 불러와서 하나씩 실행합니다.

In [ ]:
from scripts.retrieval.dense_ollama_retriever import (
    CHUNKS_PATH,
    EMBEDDINGS_PATH,
    DEFAULT_MODEL,
    DEFAULT_OLLAMA_URL,
    DEFAULT_TIMEOUT,
    load_chunks,
    check_ollama,
    embed_text,
    cosine_similarity,
    cache_matches,
    
    build_or_load_embeddings,
    search,
)

CHUNKS_PATH, EMBEDDINGS_PATH, DEFAULT_MODEL, DEFAULT_OLLAMA_URL


(WindowsPath('C:/학부연구생/wikiRAG/data/processed/survey_on_rag2_chunks.jsonl'),
 WindowsPath('C:/학부연구생/wikiRAG/data/processed/survey_on_rag2_ollama_embeddings.json'),
 'nomic-embed-text',
 'http://127.0.0.1:11434')

## 3. Ollama 연결 확인

`check_ollama()`는 로컬 Ollama 서버가 켜져 있는지 확인합니다.

정상이라면 아무 출력 없이 넘어갑니다. 에러가 나면 Ollama 앱/서버가 꺼져 있을 가능성이 큽니다.

In [3]:
check_ollama(DEFAULT_OLLAMA_URL, DEFAULT_TIMEOUT)
print("Ollama connection OK")


Ollama connection OK


## 4. chunk 파일 로드

`load_chunks()`는 JSONL chunk 파일을 읽어서 Python list로 변환합니다.

In [4]:
chunks = load_chunks(CHUNKS_PATH)

len(chunks), chunks[0].keys()


(157, dict_keys(['chunk_id', 'source', 'page', 'chunk_index', 'text']))

In [5]:
chunks[0]


{'chunk_id': 'survey_on_rag2_p1_c0',
 'source': 'survey on RAG2.pdf',
 'page': 1,
 'chunk_index': 0,
 'text': 'A Survey on RAG Meeting LLMs: Towards Retrieval-Augmented\nLarge Language Models\nWenqi Fan\nwenqifan03@gmail.com\nThe Hong Kong Polytechnic\nUniversity, HK SAR\nYujuan Ding∗\ndingyujuan385@gmail.com\nThe Hong Kong Polytechnic\nUniversity, HK SAR\nLiangbo Ning\nBigLemon1123@gmail.com\nThe Hong Kong Polytechnic\nUniversity, HK SAR\nShijie Wang\nshijie.wang@connect.polyu.hk\nThe Hong Kong Polytechnic\nUniversity, HK SAR\nHengyun Li\nneilhengyun.li@polyu.edu.hk\nThe Hong Kong Polytechnic\nUniversity, HK SAR\nDawei Yin\nyindawei@acm.org\nBaidu Inc, China\nTat-Seng Chua\ndcscts@nus.edu.sg\nNational University of Singapore,\nSingapore\nQing Li\ncsqli@comp.polyu.edu.hk\nThe Hong Kong Polytechnic\nUniversity, HK SAR\nABSTRACT\nAs one of the most advanced techniques in AI, Retrieval-Augmented\nGeneration (RAG) can offer reliable and up-to-date external knowl-\nedge, providing huge conv

## 5. query embedding 직접 확인

`embed_text()`는 입력 텍스트를 embedding vector로 바꿉니다.

embedding은 숫자 리스트이며, 전체를 출력하면 너무 길기 때문에 앞 10개 값만 확인합니다.

In [6]:
query = "What is dense retrieval?"

query_embedding = embed_text(
    text=query,
    model=DEFAULT_MODEL,
    ollama_url=DEFAULT_OLLAMA_URL,
    timeout=DEFAULT_TIMEOUT,
)

type(query_embedding), len(query_embedding), query_embedding[:10]


(list,
 768,
 [0.015451222,
  0.07191692,
  -0.17607857,
  -0.08113306,
  0.063655846,
  -0.022659209,
  0.0230572,
  0.015160481,
  -0.04256509,
  -0.021938499])

## 6. chunk embedding 직접 확인

문서 chunk도 query와 같은 방식으로 embedding합니다.

In [7]:
chunk = chunks[0]
chunk_text = chunk["text"]

chunk_embedding = embed_text(
    text=chunk_text,
    model=DEFAULT_MODEL,
    ollama_url=DEFAULT_OLLAMA_URL,
    timeout=DEFAULT_TIMEOUT,
)

chunk["chunk_id"], len(chunk_text), len(chunk_embedding), chunk_embedding[:10]


('survey_on_rag2_p1_c0',
 956,
 768,
 [-0.036146782,
  0.07931393,
  -0.16392252,
  -0.12835056,
  0.030029861,
  -0.026201116,
  0.0029501985,
  0.0076141083,
  -0.026234902,
  -0.017595593])

## 7. cosine similarity 직접 계산

query embedding과 chunk embedding의 유사도를 직접 계산합니다.

점수가 높을수록 두 텍스트가 embedding 공간에서 가깝다는 뜻입니다.

In [8]:
score = cosine_similarity(query_embedding, chunk_embedding)
score


0.5579139094005616

## 8. 여러 chunk와 직접 비교

처음 5개 chunk만 대상으로 query와의 cosine similarity를 직접 계산해봅니다.

In [9]:
sample_scores = []

for sample_chunk in chunks[:5]:
    sample_embedding = embed_text(
        text=sample_chunk["text"],
        model=DEFAULT_MODEL,
        ollama_url=DEFAULT_OLLAMA_URL,
        timeout=DEFAULT_TIMEOUT,
    )
    sample_scores.append({
        "chunk_id": sample_chunk["chunk_id"],
        "page": sample_chunk["page"],
        "score": cosine_similarity(query_embedding, sample_embedding),
        "preview": sample_chunk["text"][:160].replace("\n", " "),
    })

sorted(sample_scores, key=lambda item: item["score"], reverse=True)


[{'chunk_id': 'survey_on_rag2_p1_c3',
  'page': 1,
  'score': 0.581386134656007,
  'preview': 'Fine-tuning, In-context Learning, Prompting. ∗Corresponding author: Yujuan Ding 1This is the long version of the survey to appear at KDD 2024 [33] 1 INTRODUCTIO'},
 {'chunk_id': 'survey_on_rag2_p1_c4',
  'page': 1,
  'score': 0.5772937009669046,
  'preview': 'in external databases, can provide faithful and timely external knowledge, thereby serving vital functions in various knowledge-intensive tasks. Due to their po'},
 {'chunk_id': 'survey_on_rag2_p1_c0',
  'page': 1,
  'score': 0.5579139094005616,
  'preview': 'A Survey on RAG Meeting LLMs: Towards Retrieval-Augmented Large Language Models Wenqi Fan wenqifan03@gmail.com The Hong Kong Polytechnic University, HK SAR Yuju'},
 {'chunk_id': 'survey_on_rag2_p1_c1',
  'page': 1,
  'score': 0.5461137734135548,
  'preview': 'external knowl- edge, providing huge convenience for numerous tasks. Particularly in the era of AI-Generated Content (AIGC), 

## 9. embedding cache 확인

이미 전체 chunk embedding을 만들어둔 cache 파일이 있으면, 다음 실행부터는 다시 embedding하지 않고 재사용합니다.

In [10]:
EMBEDDINGS_PATH.exists(), EMBEDDINGS_PATH


(True,
 WindowsPath('C:/학부연구생/wikiRAG/data/processed/survey_on_rag2_ollama_embeddings.json'))

In [11]:
import json

if EMBEDDINGS_PATH.exists():
    with EMBEDDINGS_PATH.open("r", encoding="utf-8") as f:
        cache = json.load(f)
    print("model:", cache.get("model"))
    print("embedding count:", len(cache.get("embeddings", [])))
    print("cache matches:", cache_matches(cache, chunks, DEFAULT_MODEL))
    cache["embeddings"][0].keys()
else:
    print("No cache file yet")


model: nomic-embed-text
embedding count: 157
cache matches: True


## 10. 전체 embedding 로드 또는 생성

`build_or_load_embeddings()`는 cache가 맞으면 불러오고, cache가 없거나 맞지 않으면 전체 chunk embedding을 새로 만듭니다.

In [12]:
embeddings = build_or_load_embeddings(
    chunks=chunks,
    model=DEFAULT_MODEL,
    ollama_url=DEFAULT_OLLAMA_URL,
    cache_path=EMBEDDINGS_PATH,
    rebuild_cache=False,
    timeout=DEFAULT_TIMEOUT,
)

len(embeddings), embeddings[0]["chunk_id"], len(embeddings[0]["embedding"])


(157, 'survey_on_rag2_p1_c0', 768)

## 11. dense search 실행

`search()`는 query embedding을 만들고, 모든 chunk embedding과 cosine similarity를 계산한 뒤 top-k 결과를 반환합니다.

In [13]:
results = search(
    query=query,
    chunks=chunks,
    embeddings=embeddings,
    model=DEFAULT_MODEL,
    ollama_url=DEFAULT_OLLAMA_URL,
    timeout=DEFAULT_TIMEOUT,
    top_k=5,
)

results[0]


{'score': 0.7085871330195148,
 'chunk_id': 'survey_on_rag2_p4_c8',
 'source': 'survey on RAG2.pdf',
 'page': 4,
 'chunk_index': 8,
 'text': 'and data.\n3.1.2 Retrieval Granularity. Retrieval granularity denotes the re-\ntrieval unit in which the corpus is indexed, e.g., document, passage,\ntoken, or other levels like entity. For RAG, the choice of retrieval\n4'}

In [14]:
for rank, result in enumerate(results, start=1):
    print(f"[{rank}] score={result['score']:.4f} {result['chunk_id']} page={result['page']} chunk={result['chunk_index']}")
    print(result["text"][:500].replace("\n", " "))
    print()


[1] score=0.7086 survey_on_rag2_p4_c8 page=4 chunk=8
and data. 3.1.2 Retrieval Granularity. Retrieval granularity denotes the re- trieval unit in which the corpus is indexed, e.g., document, passage, token, or other levels like entity. For RAG, the choice of retrieval 4

[2] score=0.7071 survey_on_rag2_p4_c4 page=4 chunk=4
and docu- ments into continuous vector space with certain criteria, for exam- ple, semantic similarity [61]. Dense retrieval methods are usually trainable, therefore holding more flexibility and potential in adap- tation. As the key component of dense retriever, the embedding models have delicately different designs in existing RAG models. A simple design [62, 72, 165] is to directly use a part of the gener- ation model as the embedding layer of the retriever, which might be able to enhance the 

[3] score=0.7064 survey_on_rag2_p4_c3 page=4 chunk=3
117, 168, 196, 197], where passages are specifically represented as a bag of words and ranked based on term and inverse 

## 12. query 바꿔서 다시 검색

아래 query만 바꾸고 실행하면 dense retriever 결과를 계속 확인할 수 있습니다.

In [15]:
query = "How does RAG reduce hallucination?"

results = search(
    query=query,
    chunks=chunks,
    embeddings=embeddings,
    model=DEFAULT_MODEL,
    ollama_url=DEFAULT_OLLAMA_URL,
    timeout=DEFAULT_TIMEOUT,
    top_k=5,
)

for rank, result in enumerate(results, start=1):
    print(f"[{rank}] score={result['score']:.4f} {result['chunk_id']} page={result['page']} chunk={result['chunk_index']}")
    print(result["text"][:500].replace("\n", " "))
    print()


[1] score=0.5949 survey_on_rag2_p14_c1 page=14 chunk=1
By enhancing the quality of the external knowledge and tailing robust mechanisms by filtering out low-quality or unreliable information, the RA-LLM systems might produce more accurate, reliable outputs, thereby improving their effectiveness in various real-world applications. 7 CONCLUSION Retrieval-augmented generation (RAG), a cutting-edge AI tech- nique, has achieved remarkable success across various applications, including recommendation, molecule generation, protein represen- tation, and sof

[2] score=0.5943 survey_on_rag2_p2_c3 page=2 chunk=3
due to the substantial computational resources required for fine-tuning LLMs with domain-specific or the latest data. This, in turn, significantly hinders the widespread adoption of LLMs in various real-world applications. To address these limitations, recent efforts have been made to take advantage of RAG to enhance the capabilities of LLMs in var- ious tasks [ 6, 53, 62, 135], especial